# 🥉🥈🥇 Medallion Architecture — Bronze → Silver → Gold

**Complete implementation of the Medallion data architecture pattern using Fabric Delta Lake.**

**Fabric Features Showcased:**
- Delta Lake — ACID transactions, time travel, schema evolution
- V-Order — automatic columnar optimization (Fabric-exclusive)
- OPTIMIZE — compaction with bin-packing
- VACUUM — cleanup old file versions
- MERGE — upsert operations for SCD Type 1 & Type 2
- Change Data Feed — track row-level  changes
- Partition pruning — optimize query performance
- Metadata-driven transformations — all rules from config tables
- Fabric shortcuts — cross-workspace data access

**Architecture:**
```
BRONZE (Raw)          SILVER (Cleansed)        GOLD (Business)
┌──────────────┐     ┌──────────────┐        ┌──────────────┐
│ Append-only  │────▶│ SCD Type 2   │───────▶│ Star schema  │
│ Full history │     │ Deduped      │        │ Aggregates   │
│ No deletes   │     │ Validated    │        │ Metrics      │
│ Partitioned  │     │ Conformed    │        │ Power BI     │
└──────────────┘     └──────────────┘        └──────────────┘
```

**Insurance Domain Example: Policy Data**
- Bronze: Raw policy feeds from core admin system
- Silver: Cleansed policies with SCD Type 2 history
- Gold: `dim_policy`, `fact_policy_transaction`

**No SparkSession import — `spark` pre-initialized by Fabric.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Load shared utilities                                              ║
# ╚══════════════════════════════════════════════════════════════════════╝
%run 00_fabric_native_utils

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Parameters & Configuration                                         ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Layer configurations
BRONZE_LAKEHOUSE = "bronze_policy"  # Source lakehouse
SILVER_LAKEHOUSE = "silver_policy"  # Cleansed lakehouse
GOLD_LAKEHOUSE = "gold_policy"      # Business lakehouse

# Processing parameters
ENABLE_DELTA_OPTIMIZATIONS = True   # Auto-optimize after write
ENABLE_CHANGE_DATA_FEED = True       # Track row-level changes
PARTITION_COLUMN = "load_date"       # Partition by date
WATERMARK_TABLE = "metadata.processing_watermark"  # Track last processed

# Quality thresholds
MAX_NULL_PCT = 0.10  # 10% max nulls in critical columns
MAX_DUPLICATE_PCT = 0.05  # 5% max duplicates

print("✅ Configuration loaded")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 1: Bronze Layer — Ingest Raw Data                          ║
# ║  Append-only, immutable, full history                               ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import (
    current_timestamp, input_file_name, lit, col,
    to_date, year, month, dayofmonth
)

def ingest_to_bronze(source_path: str, bronze_table: str, file_format: str = "csv"):
    """Ingest raw files into Bronze layer.
    
    Bronze layer characteristics:
    - Append-only (no updates/deletes)
    - Preserve source exactly (no transformations)
    - Add metadata: ingestion_time, source_file, processing_id
    - Partition by load_date for efficient pruning
    """
    print(f"\n📥 Ingesting to Bronze: {bronze_table}")
    
    # Read source files
    if file_format == "csv":
        df_source = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(source_path)
    elif file_format == "json":
        df_source = spark.read.json(source_path)
    elif file_format == "parquet":
        df_source = spark.read.parquet(source_path)
    else:
        raise ValueError(f"Unsupported format: {file_format}")
    
    # Add Bronze metadata
    df_bronze = df_source \
        .withColumn("_ingestion_time", current_timestamp()) \
        .withColumn("_source_file", input_file_name()) \
        .withColumn("_processing_id", lit(f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}")) \
        .withColumn("load_date", to_date(current_timestamp()))
    
    # Write to Bronze Delta table (append mode)
    df_bronze.write.format("delta") \
        .mode("append") \
        .partitionBy(PARTITION_COLUMN) \
        .option("delta.enableChangeDataFeed", ENABLE_CHANGE_DATA_FEED) \
        .saveAsTable(bronze_table)
    
    row_count = df_bronze.count()
    print(f"  ✅ Ingested {row_count:,} rows → {bronze_table}")
    
    # Optimize if enabled
    if ENABLE_DELTA_OPTIMIZATIONS:
        optimize_table(bronze_table)  # From utilities
    
    return df_bronze

# Example: Ingest policy data to Bronze
# Uncomment when source files are available:
# df_bronze_policy = ingest_to_bronze(
#     source_path="Files/raw/policies/*.csv",
#     bronze_table="bronze_policy.policies_raw"
# )
print("✅ Bronze ingestion function ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 2: Silver Layer — Cleanse & Deduplicate                    ║
# ║  SCD Type 2, business rules validation, conformance                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import (
    col, row_number, desc, when, coalesce, regexp_replace,
    upper, trim, length, sha2, concat_ws
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

def bronze_to_silver_scd2(
    bronze_table: str,
    silver_table: str,
    business_key: list,
    compare_columns: list,
    cleansing_rules: dict = None
):
    """Transform Bronze to Silver with SCD Type 2.
    
    SCD Type 2 maintains full history with versioning:
    - Each change creates a new row
    - Tracks: effective_from, effective_to, is_current, row_hash
    
    Args:
        business_key: Natural key columns (e.g., ['policy_number'])
        compare_columns: Columns to detect changes
        cleansing_rules: Column transformations {col_name: function}
    """
    print(f"\n🧹 Transforming Bronze → Silver (SCD Type 2): {silver_table}")
    
    # Read Bronze
    df_bronze = spark.table(bronze_table)
    
    # 1. DEDUPLICATION — Keep latest by business key
    window_dedup = Window.partitionBy(business_key).orderBy(desc("_ingestion_time"))
    df_deduped = df_bronze \
        .withColumn("_row_num", row_number().over(window_dedup)) \
        .filter(col("_row_num") == 1) \
        .drop("_row_num")
    
    # 2. CLEANSING — Apply business rules
    df_cleansed = df_deduped
    
    if cleansing_rules:
        for col_name, rule in cleansing_rules.items():
            if rule == "uppercase":
                df_cleansed = df_cleansed.withColumn(col_name, upper(trim(col(col_name))))
            elif rule == "remove_special_chars":
                df_cleansed = df_cleansed.withColumn(col_name, regexp_replace(col(col_name), r"[^a-zA-Z0-9\s]", ""))
            elif rule == "trim":
                df_cleansed = df_cleansed.withColumn(col_name, trim(col(col_name)))
            # Add more rules as needed
    
    # 3. HASH CALCULATION — Detect changes
    df_cleansed = df_cleansed.withColumn(
        "_row_hash",
        sha2(concat_ws("|", *[coalesce(col(c).cast("string"), lit("")) for c in compare_columns]), 256)
    )
    
    # 4. SCD TYPE 2 COLUMNS
    df_cleansed = df_cleansed \
        .withColumn("effective_from", current_timestamp()) \
        .withColumn("effective_to", lit(None).cast("timestamp")) \
        .withColumn("is_current", lit(True))
    
    # 5. MERGE INTO SILVER (UPSERT)
    if spark.catalog.tableExists(silver_table):
        # Silver table exists — perform MERGE
        delta_silver = DeltaTable.forName(spark, silver_table)
        
        # Build merge condition
        merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in business_key])
        merge_condition += " AND target.is_current = true"
        
        # Perform SCD Type 2 merge
        delta_silver.alias("target").merge(
            df_cleansed.alias("source"),
            merge_condition
        ).whenMatchedUpdate(
            condition="target._row_hash != source._row_hash",
            set={
                "effective_to": "source.effective_from",
                "is_current": "false"
            }
        ).whenNotMatchedInsertAll() \
        .execute()
        
        # Insert new versions for changed records
        df_changed = df_cleansed.alias("source").join(
            delta_silver.toDF().alias("target"),
            [col(f"source.{k}") == col(f"target.{k}") for k in business_key] + \
            [col("target.is_current") == False],  # Recently expired
            "inner"
        ).where(col("source._row_hash") != col("target._row_hash")) \
         .select("source.*")
        
        if df_changed.count() > 0:
            df_changed.write.format("delta").mode("append").saveAsTable(silver_table)
        
        print(f"  ✅ MERGE completed → {silver_table}")
    
    else:
        # Initial load — create Silver table
        df_cleansed.write.format("delta") \
            .mode("overwrite") \
            .option("delta.enableChangeDataFeed", ENABLE_CHANGE_DATA_FEED) \
            .saveAsTable(silver_table)
        
        print(f"  ✅ Created Silver table → {silver_table}")
    
    # Optimize
    if ENABLE_DELTA_OPTIMIZATIONS:
        optimize_table(silver_table)
    
    return spark.table(silver_table)

# Example: Transform policy data to Silver
# df_silver_policy = bronze_to_silver_scd2(
#     bronze_table="bronze_policy.policies_raw",
#     silver_table="silver_policy.policies",
#     business_key=["policy_number"],
#     compare_columns=["policy_status", "premium_amount", "coverage_limit"],
#     cleansing_rules={
#         "policy_status": "uppercase",
#         "insured_name": "trim"
#     }
# )
print("✅ Silver SCD Type 2 function ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 3: Data Quality Validation (Bronze → Silver)               ║
# ║  Metadata-driven quality rules with scoring                         ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import col, count, when, isnan, isnull, sum as _sum

def validate_data_quality(df, quality_rules: list, table_name: str):
    """Execute data quality checks and log results.
    
    Args:
        quality_rules: [
            {"column": "policy_number", "rule": "not_null", "threshold": 0.0},
            {"column": "premium_amount", "rule": "positive", "threshold": 0.0},
            {"column": "email", "rule": "email_format", "threshold": 0.05}
        ]
    
    Returns: (df_with_quality_flag, quality_score)
    """
    print(f"\n✅ Running {len(quality_rules)} quality checks on {table_name}")
    
    total_rows = df.count()
    quality_results = []
    
    for rule in quality_rules:
        column = rule["column"]
        rule_type = rule["rule"]
        threshold = rule.get("threshold", 0.0)
        
        if rule_type == "not_null":
            null_count = df.filter(col(column).isNull()).count()
            fail_pct = null_count / total_rows
            passed = fail_pct <= threshold
            
        elif rule_type == "positive":
            negative_count = df.filter(col(column) <= 0).count()
            fail_pct = negative_count / total_rows
            passed = fail_pct <= threshold
            
        elif rule_type == "email_format":
            invalid_count = df.filter(~col(column).rlike(r"^[\w\._%+-]+@[\w\.-]+\.[A-Za-z]{2,}$")).count()
            fail_pct = invalid_count / total_rows
            passed = fail_pct <= threshold
        
        elif rule_type == "unique":
            duplicate_count = total_rows - df.select(column).distinct().count()
            fail_pct = duplicate_count / total_rows
            passed = fail_pct <= threshold
        
        else:
            fail_pct = 0.0
            passed = True
        
        quality_results.append({
            "table_name": table_name,
            "column": column,
            "rule": rule_type,
            "threshold": threshold,
            "fail_pct": fail_pct,
            "passed": passed,
            "check_time": datetime.now()
        })
        
        status = "✅" if passed else "❌"
        print(f"  {status} {column}.{rule_type}: {fail_pct:.2%} failures (threshold: {threshold:.2%})")
    
    # Calculate overall quality score
    passed_checks = sum([1 for r in quality_results if r["passed"]])
    quality_score = passed_checks / len(quality_results) if quality_results else 1.0
    
    # Log to metadata table
    from pyspark.sql import Row
    df_quality_log = spark.createDataFrame([Row(**r) for r in quality_results])
    df_quality_log.write.format("delta").mode("append").saveAsTable("metadata.data_quality_log")
    
    print(f"\n🎯 Overall Quality Score: {quality_score:.1%}")
    return df, quality_score

# Example quality rules for policy data
POLICY_QUALITY_RULES = [
    {"column": "policy_number", "rule": "not_null", "threshold": 0.0},
    {"column": "policy_number", "rule": "unique", "threshold": 0.0},
    {"column": "premium_amount", "rule": "positive", "threshold": 0.0},
    {"column": "effective_date", "rule": "not_null", "threshold": 0.01},
    {"column": "insured_email", "rule": "email_format", "threshold": 0.05},
]

print("✅ Data quality validation function ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 4: Gold Layer — Business Dimensions & Facts                ║
# ║  Star schema with conformed dimensions and aggregate facts          ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import (
    monotonically_increasing_id, col, sum as _sum, avg, count, max as _max, min as _min
)

def create_dimension_policy(silver_table: str, gold_dim_table: str):
    """Create Gold dimension table from Silver.
    
    Dimension characteristics:
    - Surrogate key (dim_policy_key)
    - Business key (policy_number)
    - Current records only (is_current = true)
    - Slowly changing attributes preserved via SCD Type 2
    """
    print(f"\n🏆 Creating Gold dimension: {gold_dim_table}")
    
    df_silver = spark.table(silver_table)
    
    # Select current records only
    df_dim = df_silver.filter(col("is_current") == True)
    
    # Add surrogate key
    df_dim = df_dim.withColumn("dim_policy_key", monotonically_increasing_id())
    
    # Select dimension attributes
    df_dim = df_dim.select(
        "dim_policy_key",
        "policy_number",  # Natural key
        "policy_status",
        "product_code",
        "insured_name",
        "coverage_amount",
        "premium_amount",
        "effective_date",
        "expiration_date",
        "agent_code",
        "region",
        # SCD columns
        "effective_from",
        "effective_to",
        "is_current"
    )
    
    # Write to Gold
    df_dim.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(gold_dim_table)
    
    optimize_table(gold_dim_table)
    print(f"  ✅ Created {df_dim.count():,} dimension records → {gold_dim_table}")
    
    return df_dim


def create_fact_policy_transactions(silver_table: str, gold_fact_table: str):
    """Create Gold fact table with policy transactions.
    
    Fact characteristics:
    - Grain: One row per policy transaction event
    - Foreign keys to dimensions
    - Measures: premium, coverage, counts
    - Partitioned by transaction_date
    """
    print(f"\n🏆 Creating Gold fact: {gold_fact_table}")
    
    df_silver = spark.table(silver_table)
    
    # Transform to fact grain (each SCD change = transaction)
    df_fact = df_silver.select(
        sha2(col("policy_number"), 256).alias("fact_key"),
        col("policy_number"),  # FK to dimension
        col("effective_from").alias("transaction_date"),  # Date dimension FK
        col("policy_status").alias("transaction_type"),  # New, Renewal, Endorsement, Cancel
        # Measures
        col("premium_amount"),
        col("coverage_amount"),
        lit(1).alias("transaction_count"),
        current_timestamp().alias("fact_created_at")
    )
    
    # Write to Gold (partitioned by date)
    df_fact.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("transaction_date") \
        .saveAsTable(gold_fact_table)
    
    optimize_table(gold_fact_table)
    print(f"  ✅ Created {df_fact.count():,} fact records → {gold_fact_table}")
    
    return df_fact


def create_aggregate_metrics(gold_fact_table: str, gold_agg_table: str, grain: list):
    """Create pre-aggregated metrics for dashboards.
    
    Args:
        grain: Aggregation level, e.g., ['region', 'product_code', 'month']
    """
    print(f"\n📊 Creating aggregate metrics: {gold_agg_table}")
    
    df_fact = spark.table(gold_fact_table)
    
    # Aggregate by grain
    df_agg = df_fact.groupBy(*grain).agg(
        _sum("premium_amount").alias("total_premium"),
        _sum("coverage_amount").alias("total_coverage"),
        count("*").alias("policy_count"),
        avg("premium_amount").alias("avg_premium"),
        _max("premium_amount").alias("max_premium"),
        _min("premium_amount").alias("min_premium")
    )
    
    # Write to Gold
    df_agg.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(gold_agg_table)
    
    optimize_table(gold_agg_table)
    print(f"  ✅ Created {df_agg.count():,} aggregate records → {gold_agg_table}")
    
    return df_agg

# Example: Create Gold layer
# dim_policy = create_dimension_policy(
#     silver_table="silver_policy.policies",
#     gold_dim_table="gold_policy.dim_policy"
# )

# fact_policy_txn = create_fact_policy_transactions(
#     silver_table="silver_policy.policies",
#     gold_fact_table="gold_policy.fact_policy_transaction"
# )

print("✅ Gold layer functions ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 5: Delta Lake Advanced Features                            ║
# ║  Time travel, VACUUM, Change Data Feed, schema evolution            ║
# ╚══════════════════════════════════════════════════════════════════════╝

def demonstrate_time_travel(table_name: str):
    """Demonstrate Delta Lake time travel.
    
    Query historical versions of data — essential for auditing.
    """
    print(f"\n⏰ Time Travel Demo: {table_name}")
    
    # Show table history
    history_df = spark.sql(f"DESCRIBE HISTORY {table_name}")
    print("\n📋 Table History:")
    display(history_df.select("version", "timestamp", "operation", "operationMetrics").limit(10))
    
    # Query previous version
    print("\n🔍 Querying version 0 (initial load):")
    df_v0 = spark.read.format("delta").option("versionAsOf", 0).table(table_name)
    print(f"   Version 0 row count: {df_v0.count():,}")
    
    # Query as of timestamp
    # df_yesterday = spark.read.format("delta") \
    #     .option("timestampAsOf", "2026-04-08") \
    #     .table(table_name)
    
    print("  ✅ Time travel queries successful")


def demonstrate_change_data_feed(table_name: str):
    """Demonstrate Change Data Feed (CDC).
    
    Track row-level changes: insert, update, delete.
    """
    print(f"\n📝 Change Data Feed Demo: {table_name}")
    
    # Read changes between versions
    changes_df = spark.read.format("delta") \
        .option("readChangeFeed", "true") \
        .option("startingVersion", 0) \
        .table(table_name)
    
    # Show change statistics
    print("\n📊 Change Statistics:")
    changes_df.groupBy("_change_type").count().show()
    
    # Changes: insert, update_preimage, update_postimage, delete
    print("  ✅ CDC tracking active")
    return changes_df


def perform_vacuum_with_retention(table_name: str, retention_hours: int = 168):
    """Vacuum old Delta files (default: 7 days retention).
    
    VACUUM removes old file versions to save storage costs.
    Retention must be >= 7 days for time travel queries.
    """
    print(f"\n🧹 Vacuuming {table_name} (retention: {retention_hours}h)")
    
    # Check current retention
    spark.sql(f"DESCRIBE DETAIL {table_name}").show()
    
    # Perform VACUUM
    spark.sql(f"VACUUM {table_name} RETAIN {retention_hours} HOURS")
    
    print(f"  ✅ VACUUM completed")


def demonstrate_schema_evolution(table_name: str):
    """Demonstrate schema evolution.
    
    Delta Lake allows adding columns without breaking pipelines.
    """
    print(f"\n🔄 Schema Evolution Demo: {table_name}")
    
    # Show current schema
    current_schema = spark.table(table_name).schema
    print("\n📋 Current Schema:")
    for field in current_schema.fields:
        print(f"   - {field.name}: {field.dataType}")
    
    # Schema evolution is enabled via:
    # .option("mergeSchema", "true")
    # or SET spark.databricks.delta.schema.autoMerge.enabled = true
    
    print("\n  💡 To evolve schema: .option('mergeSchema', 'true')")

# Run demos on a test table
# demonstrate_time_travel("silver_policy.policies")
# demonstrate_change_data_feed("silver_policy.policies")

print("✅ Delta Lake advanced features ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 6: End-to-End Medallion Pipeline Orchestration             ║
# ║  Complete Bronze → Silver → Gold flow with error handling           ║
# ╚══════════════════════════════════════════════════════════════════════╝

import mlflow

def run_medallion_pipeline(config: dict):
    """Execute complete medallion pipeline.
    
    Args:
        config: {
            "source_path": "Files/raw/policies/*.csv",
            "bronze_table": "bronze_policy.policies_raw",
            "silver_table": "silver_policy.policies",
            "gold_dim_table": "gold_policy.dim_policy",
            "gold_fact_table": "gold_policy.fact_policy_transaction",
            "business_key": ["policy_number"],
            "compare_columns": [...],
            "quality_rules": [...]
        }
    """
    print("\n" + "="*80)
    print("🚀 STARTING MEDALLION PIPELINE")
    print("="*80)
    
    # Start MLflow tracking
    with mlflow.start_run(run_name="medallion_pipeline"):
        try:
            # STEP 1: Bronze ingestion
            print("\n[1/5] Bronze Layer Ingestion")
            df_bronze = ingest_to_bronze(
                source_path=config["source_path"],
                bronze_table=config["bronze_table"]
            )
            bronze_count = df_bronze.count()
            mlflow.log_metric("bronze_rows", bronze_count)
            
            # STEP 2: Silver transformation (SCD Type 2)
            print("\n[2/5] Silver Layer Transformation")
            df_silver = bronze_to_silver_scd2(
                bronze_table=config["bronze_table"],
                silver_table=config["silver_table"],
                business_key=config["business_key"],
                compare_columns=config.get("compare_columns", []),
                cleansing_rules=config.get("cleansing_rules", {})
            )
            silver_count = df_silver.count()
            mlflow.log_metric("silver_rows", silver_count)
            
            # STEP 3: Data quality validation
            print("\n[3/5] Data Quality Validation")
            df_validated, quality_score = validate_data_quality(
                df=df_silver,
                quality_rules=config.get("quality_rules", []),
                table_name=config["silver_table"]
            )
            mlflow.log_metric("quality_score", quality_score)
            
            # STEP 4: Gold dimension
            print("\n[4/5] Gold Dimension Creation")
            df_dim = create_dimension_policy(
                silver_table=config["silver_table"],
                gold_dim_table=config["gold_dim_table"]
            )
            mlflow.log_metric("dim_rows", df_dim.count())
            
            # STEP 5: Gold fact
            print("\n[5/5] Gold Fact Creation")
            df_fact = create_fact_policy_transactions(
                silver_table=config["silver_table"],
                gold_fact_table=config["gold_fact_table"]
            )
            mlflow.log_metric("fact_rows", df_fact.count())
            
            # Log success
            mlflow.log_param("status", "SUCCESS")
            print("\n" + "="*80)
            print("✅ PIPELINE COMPLETED SUCCESSFULLY")
            print("="*80)
            print(f"\n📊 Summary:")
            print(f"   Bronze rows:    {bronze_count:,}")
            print(f"   Silver rows:    {silver_count:,}")
            print(f"   Quality score:  {quality_score:.1%}")
            print(f"   Dimension rows: {df_dim.count():,}")
            print(f"   Fact rows:      {df_fact.count():,}")
            
            return {
                "status": "SUCCESS",
                "bronze_count": bronze_count,
                "silver_count": silver_count,
                "quality_score": quality_score
            }
        
        except Exception as e:
            mlflow.log_param("status", "FAILED")
            mlflow.log_param("error", str(e))
            print(f"\n❌ PIPELINE FAILED: {e}")
            raise

# Example configuration
POLICY_PIPELINE_CONFIG = {
    "source_path": "Files/raw/policies/*.csv",
    "bronze_table": "bronze_policy.policies_raw",
    "silver_table": "silver_policy.policies",
    "gold_dim_table": "gold_policy.dim_policy",
    "gold_fact_table": "gold_policy.fact_policy_transaction",
    "business_key": ["policy_number"],
    "compare_columns": ["policy_status", "premium_amount", "coverage_amount"],
    "cleansing_rules": {
        "policy_status": "uppercase",
        "insured_name": "trim"
    },
    "quality_rules": POLICY_QUALITY_RULES
}

# Run pipeline
# result = run_medallion_pipeline(POLICY_PIPELINE_CONFIG)

print("✅ Medallion pipeline orchestration ready")

## 🎯 Summary

This notebook demonstrates **complete Medallion architecture** implementation:

### Bronze Layer
- Append-only, immutable raw data
- Metadata tracking: ingestion time, source file, processing ID
- Partitioned by load_date
- Change Data Feed enabled

### Silver Layer
- SCD Type 2 for full history
- Deduplication by business key
- Cleansing via metadata rules
- Row hash for change detection
- MERGE-based upserts
- Data quality validation

### Gold Layer
- Star schema: dimensions + facts
- Surrogate keys for dimensions
- Pre-aggregated metrics
- Optimized for Power BI Direct Lake

### Delta Lake Features Utilized
1. **V-Order** — automatic columnar optimization (Fabric-exclusive)
2. **OPTIMIZE & VACUUM** — file compaction and cleanup
3. **Time Travel** — query historical versions
4. **Change Data Feed** — track row-level changes
5. **MERGE** — efficient upserts for SCD
6. **Schema Evolution** — add columns without breaking pipelines
7. **Partition Pruning** — optimized queries

**Next Steps:**
- Apply to all insurance domains (Claims, Finance, Customers)
- Build DirectLake semantic models on Gold tables
- Set up incremental processing with watermarks
- Add Fabric Activator alerts on quality score drops